In [2]:
from kaggle_secrets import UserSecretsClient
import google.generativeai as genai

# Load API key
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Gemini_API_Key")

# Configure API
genai.configure(api_key=api_key)

# Input text
text = "vacation policy"

# Generate embedding
response = genai.embed_content(
    model="models/gemini-embedding-001",
    content=text,
    task_type="retrieval_document"
)

embedding = response["embedding"]

print("Embedding length:", len(embedding))
print("First 5 values:", embedding[:5])

Embedding length: 3072
First 5 values: [-0.02950496, 0.03353201, -0.0048776804, -0.063203365, -0.0009708868]


In [3]:
import numpy as np
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

# Load API key
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Gemini_API_Key")

# Configure Gemini
genai.configure(api_key=api_key)

# ---------------------------------------------------
# EMBEDDING FUNCTION
# ---------------------------------------------------

def get_embedding(text):

    response = genai.embed_content(
        model="models/gemini-embedding-001",
        content=text,
        task_type="retrieval_document"
    )

    return response["embedding"]

# ---------------------------------------------------
# COSINE SIMILARITY
# ---------------------------------------------------

def cosine_similarity(vec1, vec2):

    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    dot_product = np.dot(vec1, vec2)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    return dot_product / (norm1 * norm2)

# ---------------------------------------------------
# TEST PHRASES
# ---------------------------------------------------

phrases = [
    "vacation policy",
    "time off rules",
    "PTO guidelines",
    "dress code requirements"
]

# Generate embeddings
embeddings = [get_embedding(p) for p in phrases]

# Base phrase
base = embeddings[0]

print(f'Comparing "{phrases[0]}" with:\n')

for i, phrase in enumerate(phrases[1:], 1):

    similarity = cosine_similarity(base, embeddings[i])

    print(f'{phrase:30} Similarity: {similarity:.4f}')

Comparing "vacation policy" with:

time off rules                 Similarity: 0.8962
PTO guidelines                 Similarity: 0.8699
dress code requirements        Similarity: 0.7716


In [6]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 92.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0

In [7]:
import os
import chromadb

from chromadb.utils import embedding_functions
from kaggle_secrets import UserSecretsClient

# ---------------------------------------------------
# LOAD GEMINI API KEY
# ---------------------------------------------------

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Gemini_API_Key")

# Store in environment variable
os.environ["GEMINI_API_KEY"] = api_key

# ---------------------------------------------------
# CREATE CHROMADB CLIENT
# ---------------------------------------------------

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

# ---------------------------------------------------
# EMBEDDING FUNCTION
# ---------------------------------------------------

# NOTE:
# ChromaDB ka built-in OpenAIEmbeddingFunction
# Gemini ke sath directly compatible nahi hota.
# Is liye filhal default embedding function use kar rahe hain.

embedding_function = embedding_functions.DefaultEmbeddingFunction()

# ---------------------------------------------------
# CREATE / GET COLLECTION
# ---------------------------------------------------

collection = chroma_client.get_or_create_collection(
    name="company_docs",
    embedding_function=embedding_function,
    metadata={
        "description": "Company policy documents"
    }
)

# ---------------------------------------------------
# INFO
# ---------------------------------------------------

print(f"Collection: {collection.name}")
print(f"Count: {collection.count()}")

Collection: company_docs
Count: 0


In [19]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader

DATA_PATH = "/kaggle/input/datasets/waynaali/company-docs/company_docs"

print("Files in path:")
print(os.listdir(DATA_PATH))

Files in path:
['it_policy.txt', 'benefits.txt', 'hr_policy.txt']


In [20]:
# =========================================================
# 1. INSTALL (if needed)
# =========================================================
# !pip install chromadb langchain langchain-community langchain-text-splitters

# =========================================================
# 2. IMPORTS
# =========================================================
import os
import chromadb

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# =========================================================
# 3. DATA PATH
# =========================================================
DATA_PATH = "/kaggle/input/datasets/waynaali/company-docs"

print("📂 Checking files in dataset:")
print(os.listdir(DATA_PATH))

# =========================================================
# 4. LOAD DOCUMENTS
# =========================================================
loader = DirectoryLoader(
    DATA_PATH,
    glob="**/*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print("Loaded documents:", len(documents))

if len(documents) == 0:
    raise ValueError("❌ No documents loaded. Check dataset path or file type.")

# =========================================================
# 5. SPLIT INTO CHUNKS
# =========================================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Created chunks:", len(chunks))

if len(chunks) == 0:
    raise ValueError("❌ No chunks created.")

# =========================================================
# 6. CHROMADB SETUP
# =========================================================
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="company_docs"
)

print("Collection:", collection.name)
print("Current Count:", collection.count())

# =========================================================
# 7. ADD DATA TO CHROMA (SAFE)
# =========================================================
if collection.count() == 0 and len(chunks) > 0:

    collection.add(
        documents=[c.page_content for c in chunks],
        ids=[f"doc_{i}" for i in range(len(chunks))],
        metadatas=[{"source": c.metadata.get("source", "unknown")} for c in chunks]
    )

    print(f"✅ Added {len(chunks)} chunks to ChromaDB")

else:
    print(f"⚠️ Already exists: {collection.count()} documents")

# =========================================================
# 8. VECTOR SEARCH
# =========================================================
def vector_search(query, n_results=3):

    return collection.query(
        query_texts=[query],
        n_results=n_results
    )

# =========================================================
# 9. KEYWORD SEARCH
# =========================================================
def keyword_search(query, chunks, top_k=3):

    query_words = query.lower().split()
    scored = []

    for chunk in chunks:
        text = chunk.page_content.lower()
        score = sum(text.count(word) for word in query_words)

        if score > 0:
            scored.append((score, chunk))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [c for s, c in scored[:top_k]]

# =========================================================
# 10. TEST
# =========================================================
query = "PTO policy"

print("\n================ KEYWORD SEARCH ================")
kw_results = keyword_search(query, chunks, top_k=2)
print("Found:", len(kw_results))

print("\n================ SEMANTIC SEARCH ================")
sem_results = vector_search(query, n_results=2)

print("Found:", len(sem_results["documents"][0]))
print("Top result:", sem_results["documents"][0][0][:200])

📂 Checking files in dataset:
['company_docs']
Loaded documents: 3
Created chunks: 5
Collection: company_docs
Current Count: 0
✅ Added 5 chunks to ChromaDB

================ KEYWORD SEARCH ================
Found: 2

================ SEMANTIC SEARCH ================
Found: 2
Top result: Employee Handbook - HR Policies  Vacation Policy: All full-time employees receive 15 days of paid vacation per year. Vacation days accrue monthly and can be used after 90 days.  Remote Work Policy: Em


In [10]:
# =========================================================
# VECTOR SEARCH FUNCTION
# =========================================================

def vector_search(query, n_results=3):
    """
    Search using semantic similarity
    """

    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    return results


# =========================================================
# TEST QUERIES
# =========================================================

test_queries = [

    "time off policy",     # should match vacation policy

    "WFH guidelines",      # should match remote work

    "maternity leave"      # should match parental leave
]

# =========================================================
# RUN SEARCHES
# =========================================================

for query in test_queries:

    print("\n" + "=" * 60)

    print(f"Query: {query}")

    print("=" * 60)

    results = vector_search(
        query,
        n_results=2
    )

    for i, doc in enumerate(
        results["documents"][0]
    ):

        distance = (
            results["distances"][0][i]
        )

        print(
            f"\nResult {i+1} "
            f"(distance: {distance:.4f}):"
        )

        print(doc[:200] + "...")


Query: time off policy


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 103MiB/s] 



Query: WFH guidelines

Query: maternity leave


In [11]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0


In [21]:
import google.generativeai as genai

genai.configure(api_key="Gemini_API_Key")

llm = genai.GenerativeModel("gemini-2.5-flash")

def semantic_rag(query, n_results=3):

    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    if not results["documents"][0]:
        return "No relevant information found."

    context = "\n\n---\n\n".join(results["documents"][0])

    prompt = f"""
You are a helpful HR assistant.
Answer ONLY from context.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.generate_content(prompt)
    return response.text

In [22]:
# From Week 10 - keyword search

def keyword_search(query, chunks, top_k=3):
    query_lower = query.lower()
    scored = []

    for chunk in chunks:
        score = sum(chunk.page_content.lower().count(word) for word in query_lower.split())

        if score > 0:
            scored.append((score, chunk))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [c for s, c in scored[:top_k]]


# Compare on synonym query
query = "PTO policy"  # Uses 'PTO', docs say 'vacation'

print("KEYWORD SEARCH:")
kw_results = keyword_search(query, chunks, top_k=2)
print(f"Found: {len(kw_results)} results")

print("\nSEMANTIC SEARCH:")
sem_results = vector_search(query, n_results=2)
print(f"Found: {len(sem_results['documents'][0])} results")
print(f"Top result: {sem_results['documents'][0][0][:100]}...")

KEYWORD SEARCH:
Found: 2 results

SEMANTIC SEARCH:
Found: 2 results
Top result: Employee Handbook - HR Policies  Vacation Policy: All full-time employees receive 15 days of paid va...
